# Detección de phishing con IA: modelo híbrido (reglas + aprendizaje automático)
Proyecto de Inteligencia Artificial y Sistemas Expertos, Universidad Nihon Gakko.

**Antes de ejecutar:** subir a Colab (panel izquierdo, icono de carpeta) los archivos
`Nazario.csv` y `SpamAssasin.csv` de la colección Zenodo 10.5281/zenodo.8339691.

In [ ]:
import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)
warnings.filterwarnings("ignore")

SEED = 42
PHISHING_FILE = "Nazario.csv"        # correos de phishing
LEGIT_FILE = "SpamAssasin.csv"       # correos legítimos (se usan solo los de label == 0)
ALPHA = 0.3                          # peso de las reglas en el modelo híbrido (0 a 1)
THRESHOLD_RULES = 3                  # puntaje mínimo para que las reglas marquen phishing

## 1. Carga y preparación de datos

In [ ]:
def leer(ruta):
    """Lee el CSV; si falla por comillas mal cerradas, usa una lectura tolerante."""
    try:
        return pd.read_csv(ruta)
    except Exception as e:
        print("Aviso: lectura estándar falló en", ruta, "->", str(e)[:80])
        return pd.read_csv(ruta, engine="python", on_bad_lines="skip")

phish = leer(PHISHING_FILE)
legit = leer(LEGIT_FILE)

print("Nazario  -> filas:", len(phish), "| etiquetas:", phish["label"].value_counts().to_dict())
print("Legítimos-> filas:", len(legit), "| etiquetas:", legit["label"].value_counts().to_dict())

phish = phish[phish["label"] == 1].copy()
legit = legit[legit["label"] == 0].copy()          # solo correos NO spam
phish["label"], legit["label"] = 1, 0

# Balanceo de clases: mismo número de correos en cada una
n = min(len(phish), len(legit))
df = pd.concat([phish.sample(n, random_state=SEED), legit.sample(n, random_state=SEED)])
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

for col in ["subject", "body"]:
    df[col] = df[col].fillna("").astype(str)
df["urls"] = pd.to_numeric(df.get("urls", 0), errors="coerce").fillna(0)
df["text"] = df["subject"] + " " + df["body"]
df = df[df["text"].str.strip() != ""].reset_index(drop=True)

print("\nConjunto final:", len(df), "correos |", df["label"].value_counts().to_dict())

## 2. Partición entrenamiento / prueba (80 % / 20 %)

In [ ]:
X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    df["text"], df["label"], df, test_size=0.20, random_state=SEED, stratify=df["label"])
print("Entrenamiento:", len(X_train), "| Prueba:", len(X_test))

## 3. Capa de reglas heurísticas (componente de sistema experto)
Cada regla suma 1 punto. Si el puntaje llega al umbral, el correo se marca como phishing.

In [ ]:
URGENCIA = r"urgent|immediately|within 24|expire|suspend|deactivat|final notice|act now|urgente|inmediatamente|suspendid|expira"
CREDENCIALES = r"password|username|login|log in|sign in|verify your|confirm your|credential|contraseña|verifique|confirme"
ACCION = r"click here|click the link|follow the link|update your|haga clic|actualice"
CUENTA = r"your account|mailbox|helpdesk|help desk|it-service|security alert|su cuenta|buzón"
IP_URL = r"https?://\d{1,3}(\.\d{1,3}){3}"

REGLAS = {
    "R1 lenguaje de urgencia": lambda t, u: bool(re.search(URGENCIA, t)),
    "R2 pide credenciales": lambda t, u: bool(re.search(CREDENCIALES, t)),
    "R3 invita a hacer clic/actualizar": lambda t, u: bool(re.search(ACCION, t)),
    "R4 menciona cuenta/buzón/soporte": lambda t, u: bool(re.search(CUENTA, t)),
    "R5 contiene enlaces": lambda t, u: u > 0 or "http" in t,
    "R6 enlace con dirección IP": lambda t, u: bool(re.search(IP_URL, t)),
}

def puntaje_reglas(texto, urls):
    t = texto.lower()
    return sum(int(f(t, urls)) for f in REGLAS.values())

df_test = df_test.copy()
df_test["score_reglas"] = [puntaje_reglas(t, u) for t, u in zip(df_test["text"], df_test["urls"])]
pred_reglas = (df_test["score_reglas"] >= THRESHOLD_RULES).astype(int).values

## 4. Modelos de aprendizaje automático

In [ ]:
def crear_pipeline(clf):
    return Pipeline([("tfidf", TfidfVectorizer(stop_words="english", max_features=5000,
                                               min_df=2, sublinear_tf=True)),
                     ("clf", clf)])

modelos = {
    "Naive Bayes": MultinomialNB(),
    "Regresión Logística": LogisticRegression(max_iter=1000, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
}

def metricas(y_true, y_pred):
    return {"Exactitud": accuracy_score(y_true, y_pred),
            "Precisión": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0)}

filas, entrenados, probas = [], {}, {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for nombre, clf in modelos.items():
    pipe = crear_pipeline(clf).fit(X_train, y_train)
    entrenados[nombre] = pipe
    probas[nombre] = pipe.predict_proba(X_test)[:, 1]
    m = metricas(y_test, pipe.predict(X_test))
    f1_cv = cross_val_score(crear_pipeline(clf), X_train, y_train, cv=cv, scoring="f1")
    m["F1 validación cruzada (media ± desv.)"] = f"{f1_cv.mean():.3f} ± {f1_cv.std():.3f}"
    filas.append({"Modelo": nombre, **m})

filas.append({"Modelo": f"Solo reglas (umbral {THRESHOLD_RULES})", **metricas(y_test, pred_reglas),
              "F1 validación cruzada (media ± desv.)": "n/a"})

## 5. Modelo híbrido: probabilidad del mejor modelo + puntaje de reglas

In [ ]:
mejor = max(modelos, key=lambda k: f1_score(y_test, (probas[k] >= 0.5).astype(int)))
max_reglas = len(REGLAS)
p_hibrido = (1 - ALPHA) * probas[mejor] + ALPHA * (df_test["score_reglas"].values / max_reglas)
pred_hibrido = (p_hibrido >= 0.5).astype(int)
filas.append({"Modelo": f"Híbrido ({mejor} + reglas, α={ALPHA})", **metricas(y_test, pred_hibrido),
              "F1 validación cruzada (media ± desv.)": "n/a"})

resultados = pd.DataFrame(filas)
num = ["Exactitud", "Precisión", "Recall", "F1"]
resultados[num] = resultados[num].round(4)
resultados.to_csv("resultados_modelos.csv", index=False)
print(resultados.to_string(index=False))

## 6. Gráficos para el informe

In [ ]:
# 6.1 Comparación de modelos
ax = resultados.set_index("Modelo")[num].plot(kind="bar", figsize=(10, 5), ylim=(0, 1.05), rot=15)
ax.set_title("Comparación de desempeño en el conjunto de prueba")
ax.set_ylabel("Valor de la métrica"); ax.legend(loc="lower right")
plt.tight_layout(); plt.savefig("fig1_comparacion_modelos.png", dpi=200); plt.show()

# 6.2 Matriz de confusión del modelo híbrido
cm = confusion_matrix(y_test, pred_hibrido)
ConfusionMatrixDisplay(cm, display_labels=["Legítimo", "Phishing"]).plot(cmap="Blues", values_format="d")
plt.title("Matriz de confusión: modelo híbrido")
plt.tight_layout(); plt.savefig("fig2_matriz_confusion_hibrido.png", dpi=200); plt.show()
print("Falsos positivos (legítimos marcados como phishing):", cm[0, 1])
print("Falsos negativos (phishing no detectados):", cm[1, 0])

# 6.3 Palabras más influyentes (Regresión Logística)
lr = entrenados["Regresión Logística"]
vocab = np.array(lr.named_steps["tfidf"].get_feature_names_out())
coef = lr.named_steps["clf"].coef_[0]
top = np.argsort(coef)
palabras = list(vocab[top[:10]]) + list(vocab[top[-10:]])
valores = list(coef[top[:10]]) + list(coef[top[-10:]])
plt.figure(figsize=(8, 6))
plt.barh(palabras, valores, color=["tab:green"] * 10 + ["tab:red"] * 10)
plt.title("Palabras con mayor peso (rojo: phishing, verde: legítimo)")
plt.tight_layout(); plt.savefig("fig3_palabras_influyentes.png", dpi=200); plt.show()

## 7. Ejemplos de falsos negativos (para analizar limitaciones)

In [ ]:
fn = df_test[(y_test.values == 1) & (pred_hibrido == 0)]
print(fn[["subject"]].head(5).to_string(index=False))

**Descargar:** `resultados_modelos.csv`, `fig1_comparacion_modelos.png`,
`fig2_matriz_confusion_hibrido.png`, `fig3_palabras_influyentes.png` (panel de archivos de Colab).

## 8. Guardar el modelo (para el prototipo `prototipo/verificador.py`)

In [ ]:
import joblib
joblib.dump({"pipeline": entrenados[mejor], "alpha": ALPHA, "mejor": mejor, "max_reglas": len(REGLAS)},
            "modelo_hibrido.joblib")
print("Modelo guardado: modelo_hibrido.joblib (", mejor, ")")